In [ ]:
# Imports
import pandas as pd
from datetime import date

from google.cloud import bigquery

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

import meteostat as ms

In [ ]:



# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

PROJECT_ID = "pacey32-agency"

client = bigquery.Client(project=PROJECT_ID)

geolocator = Nominatim(
    user_agent="pacey32-hockey-climate",
    timeout=10
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1
)


# ------------------------------------------------------------------
# Read cities from BigQuery
# ------------------------------------------------------------------

sql = """
SELECT DISTINCT venueLocation
FROM `pacey32-agency.Team.TeamList`
ORDER BY venueLocation
"""

cities = client.query(sql).to_dataframe()


# ------------------------------------------------------------------
# Geocode
# ------------------------------------------------------------------

def get_lat_lon(city):

    location = geocode(city)

    if location:
        return pd.Series(
            [location.latitude, location.longitude]
        )

    return pd.Series([None, None])


cities[["latitude", "longitude"]] = (
    cities["venueLocation"]
    .apply(get_lat_lon)
)


# ------------------------------------------------------------------
# Meteostat
# ------------------------------------------------------------------

START_DATE = date(1991, 1, 1)
END_DATE = date(2020, 12, 31)

weather = []

for _, row in cities.iterrows():

    point = ms.Point(
        row.latitude,
        row.longitude
    )

    monthly = (
        ms.Monthly(point, START_DATE, END_DATE)
        .fetch()
        .reset_index()
    )

    monthly["Month"] = pd.to_datetime(
        monthly["time"]
    ).dt.month

    monthly = (
        monthly
        .groupby("Month", as_index=False)
        .agg(
            AvgTemp=("tavg", "mean"),
            MinTemp=("tmin", "mean"),
            MaxTemp=("tmax", "mean"),
            RainMM=("prcp", "mean"),
            SunshineMinutes=("tsun", "mean"),
        )
    )

    monthly["venueLocation"] = row.venueLocation
    monthly["latitude"] = row.latitude
    monthly["longitude"] = row.longitude

    weather.append(monthly)


weather = pd.concat(weather, ignore_index=True)


print(weather.head())